In [23]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [24]:
def gen_spikes(rate_tensor):
    random_vals = torch.rand_like(rate_tensor)
    spikes = (random_vals < rate_tensor).float()
    
    return spikes

In [25]:
class ATan(torch.autograd.Function):
    
    @staticmethod
    def forward(ctx, input_, alpha):
        ctx.save_for_backward(input_)
        ctx.alpha = alpha
        out = (input_ > 0).float()
        return out

    @staticmethod
    def backward(ctx, grad_output):
        (input_,) = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad = (
            ctx.alpha
            / 2
            / (1 + (torch.pi / 2 * ctx.alpha * input_).pow_(2))
            * grad_input
        )
        return grad, None


class LIF:


    def __init__(self, time_step, R, C, V_threshold):

        self.time_step = time_step
        self.threshold = V_threshold
        self.R = R
        self.C = C
        self.tau = R * C

    def __call__(self, *args, **kwds):
        return self.forward(*args, **kwds)

    # neuron function (non-leaky)
    def forward(self, I, V_prev):

        # fire and reset mechanisms
        #spk_out = (V_prev >= self.threshold).float()
        spk_out = ATan.apply(V_prev - self.threshold, 25.0)

        V = V_prev + (self.time_step / self.tau) * (-V_prev + self.R * I) - spk_out * self.threshold

        return spk_out, V

In [26]:
class Net(nn.Module):
    # define all the constants
    R_m = 5.1  # Membrane resistance in MΩ
    C_m = 5e-3  # Membrane capacitance in pF
    time_step = 1e-3  # Time step in ms

    V_threshold = 1  # Threshold potential for firing a spike in mV


    hidden_features = 128

    def __init__(self, timesteps, in_features, out_features):
        super(Net, self).__init__()

        self.timesteps = timesteps
        self.in_features = in_features
        self.out_features = out_features

        self.fc1 = nn.Linear(in_features, self.hidden_features)
        self.fc2 = nn.Linear(self.hidden_features, out_features)

        self.lif1 = LIF(time_step=self.time_step, R=self.R_m, C=self.C_m, V_threshold=self.V_threshold)
        self.lif2 = LIF(time_step=self.time_step, R=self.R_m, C=self.C_m, V_threshold=self.V_threshold)


    def forward(self, x):
        #x = x.view(-1, 28 * 28)  # Flatten
        
        # initialize membrane potentials with [batch_size, num_neurons]
        mem1 = torch.zeros(x.size(0), self.hidden_features)
        mem2 = torch.zeros(x.size(0), self.out_features)

        # Record the spikes from the last layer
        spk2_rec = []
        mem2_rec = []
        
        for t in range(self.timesteps):

            cur1 = self.fc1(x) # in : [batch_size, in_features] out : [batch_size, hidden_features]
            spk1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)

            spk2_rec.append(spk2)
            mem2_rec.append(mem2)
        

        return torch.stack(spk2_rec, dim=1), torch.stack(mem2_rec, dim=1)

In [27]:
learning_rate = 0.001

num_steps = 25
in_features = 5
out_features = 10
num_batches = 1

num_epochs = 1

# Load data
transform = transforms.ToTensor()
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)


In [28]:

model = Net(timesteps=num_steps, in_features=784, out_features=10)

optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()


In [33]:
# single forward pass, validate network and inspect shapes
with torch.no_grad():
    total = 0
    correct = 0
    model.eval()

    
    for data, targets in test_loader:
        print(f"Input shape: {data.shape}") # Should be [batch_size, 1, 28, 28]

        data = data.view(-1, 28 * 28)  # Flatten

        print(f"Flattened input shape: {data.shape}") # Should be [batch_size, 784]
        
        spk_out, _ = model(data)

        print(f"Output spike tensor shape: {spk_out.shape}")  # Should be [batch_size, timesteps, out_features]


        spike_counts = spk_out.sum(dim=1)

        print(f"Spike counts shape: {spike_counts.shape}")  # Should be [batch_size, out_features]

        predictions = spike_counts.argmax(dim=1)

        print(f"Predictions shape: {predictions.shape}")  # Should be [batch_size]
        break

Input shape: torch.Size([1000, 1, 28, 28])
Flattened input shape: torch.Size([1000, 784])
Output spike tensor shape: torch.Size([1000, 25, 10])
Spike counts shape: torch.Size([1000, 10])
Predictions shape: torch.Size([1000])


In [ ]:
loss_hist = [] # record loss over iterations
acc_hist = [] # record accuracy over iterations

for epoch in range(num_epochs):
    for i, (data, targets) in enumerate(train_loader):

        data = data.view(-1, 28 * 28)  # Flatten

        model.train()

        spk_out, _ = model(data)
        
        logits = spk_out.sum(dim=1)

        loss = nn.CrossEntropyLoss()(logits, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_hist.append(loss.item())

        # print every 25 iterations
        if i % 25 == 0:
            model.eval()
            print(f"Epoch {epoch}, Iteration {i} \nTrain Loss: {loss.item():.2f}")
            
            # check accuracy on a single batch
            spike_counts = spk_out.sum(dim=1)
            predictions = spike_counts.argmax(dim=1)
            correct += (predictions == targets).sum().item()
            total += targets.size(0)
        
            acc = correct / total
            acc_hist.append(acc)
            print(f"Accuracy: {acc * 100:.2f}%\n")

Epoch 0, Iteration 0 
Train Loss: 2.30
Accuracy: 7.81%

Epoch 0, Iteration 25 
Train Loss: 2.18
Accuracy: 12.50%

Epoch 0, Iteration 50 
Train Loss: 2.17
Accuracy: 16.15%

Epoch 0, Iteration 75 
Train Loss: 2.07
Accuracy: 19.14%

Epoch 0, Iteration 100 
Train Loss: 2.15
Accuracy: 20.62%

Epoch 0, Iteration 125 
Train Loss: 2.13
Accuracy: 21.35%

Epoch 0, Iteration 150 
Train Loss: 2.17
Accuracy: 21.21%

Epoch 0, Iteration 175 
Train Loss: 2.00
Accuracy: 22.66%

Epoch 0, Iteration 200 
Train Loss: 2.18
Accuracy: 22.22%

Epoch 0, Iteration 225 
Train Loss: 2.19
Accuracy: 22.19%

Epoch 0, Iteration 250 
Train Loss: 2.21
Accuracy: 21.73%

Epoch 0, Iteration 275 
Train Loss: 2.21
Accuracy: 21.48%

Epoch 0, Iteration 300 
Train Loss: 2.22
Accuracy: 21.03%

Epoch 0, Iteration 325 
Train Loss: 2.18
Accuracy: 20.87%

Epoch 0, Iteration 350 
Train Loss: 2.23
Accuracy: 20.83%

Epoch 0, Iteration 375 
Train Loss: 2.21
Accuracy: 21.00%

Epoch 0, Iteration 400 
Train Loss: 2.13
Accuracy: 21.51%

Epo

In [32]:
with torch.no_grad():
    total = 0
    correct = 0
    model.eval()

    for data, targets in test_loader:
        data = data.view(-1, 28 * 28)  # Flatten
        
        spk_out, _ = model(data)

        spike_counts = spk_out.sum(dim=1)
        predictions = spike_counts.argmax(dim=1)

        correct += (predictions == targets).sum().item()
        total += targets.size(0)

    print(f"Accuracy: {100 * correct / total:.2f}%")

Accuracy: 11.21%
